In [1]:
import pandas as pd
import numpy as np

TARGETS = ["Theta"]

SETS = ["Train_1", "Train_2", "Val_1", "Val_2", "Test_1", "Test_2", "LSG_1", "LSG_2"]

results = pd.read_excel("resultados.xlsx")


In [2]:

results = pd.read_excel("resultados.xlsx")

best_models_tables = {}

N = 10  # top modelos

for target in TARGETS:

    # 🔹 nomes das colunas
    r2_cols = {s: f"R2_{s}_{target}" for s in SETS}

    # 🔹 garantir que só usamos colunas existentes
    valid_r2 = {k: v for k, v in r2_cols.items() if v in results.columns}


    df = results.copy()# 🔹 pegar todas as colunas de R2 válidas
    r2_all_cols = list(valid_r2.values())

    # 🔹 remover linhas onde QUALQUER R2 < 0
    df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    
    train_cols = [v for k, v in valid_r2.items() if "Train" in k]
    test_cols  = [v for k, v in valid_r2.items() if "Test" in k]
    val_cols   = [v for k, v in valid_r2.items() if "Val" in k]
    test2_cols  = [v for k, v in valid_r2.items() if "LSG" in k]


    df["R2_train_mean"] = df[train_cols].mean(axis=1)
    df["R2_test_mean"]  = df[test_cols].mean(axis=1)
    df["R2_val"]        = df[val_cols].mean(axis=1)
    df["R2_test2_mean"]        = df[test2_cols].mean(axis=1)

    # =========================
    # 🔹 SCORE
    # =========================
    
    w_val = 0.3
    w_test = 0.2
    w_test2 = 0.3
    w_train = 0.2

    metric_cols = list(r2_cols.values())
    df["R2_std"] = df[metric_cols].std(axis=1)

    df["Score"] = (
        w_val * df["R2_val"] +
        w_test * df["R2_test_mean"] +
        w_test2 * df["R2_test2_mean"] +
        w_train * df["R2_train_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    
    df_sorted = df.sort_values(by="Score", ascending=False)

    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_val", "R2_test_mean", "R2_test2_mean", "R2_train_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    # 🔹 organizar colunas
    metric_cols = []

    for s in SETS:
        if s in valid_r2:
            metric_cols.append(valid_r2[s])

    final_cols = ["model", "Neurons"] + metric_cols + [
        "R2_val", "R2_test_mean", "R2_test2_mean", "R2_train_mean", "Score"
    ]

    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - Theta


,model,Neurons,R2_val,R2_test_mean,R2_test2_mean,R2_train_mean,Score
13,model_arch16_r0.01_Ld0.7_Lp0.3_seed5410,[16],0.809340,0.653477,0.664507,0.633796,0.683362
129,model_arch16-8_r0.9_Ld0.3_Lp0.7_seed6545,"[16, 8]",0.732049,0.643926,0.634366,0.772953,0.675965
5,model_arch16_r0.9_Ld0.3_Lp0.7_seed3977,[16],0.839917,0.525747,0.580925,0.803210,0.674576
154,model_arch32-16_r0.01_Ld0.7_Lp0.3_seed6545,"[32, 16]",0.771366,0.638973,0.565995,0.747880,0.657560
1,model_arch16_r0.01_Ld0.3_Lp0.7_seed356,[16],0.886481,0.414159,0.552353,0.830877,0.654148
0,model_arch16_r0.01_Ld0.3_Lp0.7_seed3977,[16],0.838421,0.558320,0.430442,0.802413,0.627299
255,model_arch32-16-8_r0.9_Ld0.7_Lp0.3_seed3977,"[32, 16, 8]",0.789932,0.242658,0.701076,0.691588,0.609758
164,model_arch64-32_r0.01_Ld0.3_Lp0.7_seed6545,"[64, 32]",0.788973,0.386318,0.629864,0.598294,0.600135
155,model_arch32-16_r0.9_Ld0.7_Lp0.3_seed3977,"[32, 16]",0.779927,0.228723,0.612941,0.511423,0.542766
136,model_arch16-8_r0.9_Ld0.7_Lp0.3_seed356,"[16, 8]",0.686935,0.389448,0.419699,0.505860,0.483912



📊 MÉTRICAS COMPLETAS - TOP 10 (Theta)


,model,Neurons,R2_Train_1_Theta,R2_Train_2_Theta,R2_Val_1_Theta,R2_Val_2_Theta,R2_Test_1_Theta,R2_Test_2_Theta,R2_LSG_1_Theta,R2_LSG_2_Theta,R2_val,R2_test_mean,R2_test2_mean,R2_train_mean,Score
13,model_arch16_r0.01_Ld0.7_Lp0.3_seed5410,[16],0.774572,0.493019,0.939161,0.679519,0.591210,0.715744,0.484367,0.844647,0.809340,0.653477,0.664507,0.633796,0.683362
129,model_arch16-8_r0.9_Ld0.3_Lp0.7_seed6545,"[16, 8]",0.821504,0.724402,0.893724,0.570374,0.475021,0.812831,0.448839,0.819893,0.732049,0.643926,0.634366,0.772953,0.675965
5,model_arch16_r0.9_Ld0.3_Lp0.7_seed3977,[16],0.819413,0.787007,0.898761,0.781073,0.669269,0.382226,0.672175,0.489675,0.839917,0.525747,0.580925,0.803210,0.674576
154,model_arch32-16_r0.01_Ld0.7_Lp0.3_seed6545,"[32, 16]",0.849688,0.646071,0.884063,0.658670,0.392580,0.885366,0.358093,0.773898,0.771366,0.638973,0.565995,0.747880,0.657560
1,model_arch16_r0.01_Ld0.3_Lp0.7_seed356,[16],0.830978,0.830777,0.890258,0.882704,0.609243,0.219075,0.788692,0.316014,0.886481,0.414159,0.552353,0.830877,0.654148
0,model_arch16_r0.01_Ld0.3_Lp0.7_seed3977,[16],0.877719,0.727107,0.848808,0.828034,0.750613,0.366026,0.694678,0.166207,0.838421,0.558320,0.430442,0.802413,0.627299
255,model_arch32-16-8_r0.9_Ld0.7_Lp0.3_seed3977,"[32, 16, 8]",0.802166,0.581010,0.898698,0.681166,0.242655,0.242660,0.656437,0.745714,0.789932,0.242658,0.701076,0.691588,0.609758
164,model_arch64-32_r0.01_Ld0.3_Lp0.7_seed6545,"[64, 32]",0.646334,0.550253,0.717372,0.860574,0.672997,0.099639,0.695702,0.564027,0.788973,0.386318,0.629864,0.598294,0.600135
155,model_arch32-16_r0.9_Ld0.7_Lp0.3_seed3977,"[32, 16]",0.476183,0.546663,0.669986,0.889867,0.230015,0.227431,0.728664,0.497218,0.779927,0.228723,0.612941,0.511423,0.542766
136,model_arch16-8_r0.9_Ld0.7_Lp0.3_seed356,"[16, 8]",0.356245,0.655475,0.588588,0.785283,0.573130,0.205766,0.791659,0.047739,0.686935,0.389448,0.419699,0.505860,0.483912


In [3]:
final_table.to_excel("BestModels.xlsx")